# Build Modules
This notebook creates all the modular `.py` files for the skincare recommendation pipeline.
Run all cells to create/update the files in the Google Drive shared folder.

**Structure created:**
```
skincare_project/
├── agents/
│   ├── __init__.py
│   ├── skin_profile.py
│   ├── retrieval.py
│   ├── conflict_checker.py
│   └── routine_builder.py
├── pipeline.py
└── data_loader.py
```

## Setup: Mount Drive & Create Folders

In [55]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Create agents folder if it doesn't exist
BASE_DIR = "/content/drive/MyDrive/skincare_project"
os.makedirs(f"{BASE_DIR}/agents", exist_ok=True)
print("Folders ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folders ready.


## agents/__init__.py
Makes the agents folder a proper Python package so we can import from it.

In [56]:
%%writefile /content/drive/MyDrive/skincare_project/agents/__init__.py
# Skincare Recommendation System - Agents Package

Overwriting /content/drive/MyDrive/skincare_project/agents/__init__.py


## data_loader.py
Handles all data loading, cleaning, merging, and ChromaDB indexing.

In [57]:
%%writefile /content/drive/MyDrive/skincare_project/data_loader.py
import pandas as pd
import ast
import chromadb

DATA_DIR = "/content/drive/MyDrive/skincare_project/data"


def parse_ingredients_list(ingred_str):
    """Parse ingredient list from string representation of Python list."""
    try:
        ingredients = ast.literal_eval(ingred_str)
        return ", ".join([i.strip().lower() for i in ingredients])
    except (ValueError, SyntaxError, TypeError):
        return ""


def extract_brand(name):
    """Extract brand name from product name for lookfantastic products."""
    brand_mapping = {
        "The Ordinary": "The Ordinary",
        "CeraVe": "CeraVe",
        "AMELIORATE": "AMELIORATE",
        "La Roche-Posay": "La Roche-Posay",
        "Peter Thomas Roth": "Peter Thomas Roth",
        "Paula's Choice": "Paula's Choice",
        "Drunk Elephant": "Drunk Elephant",
    }
    if pd.isna(name):
        return "Unknown"
    for brand in brand_mapping:
        if brand.lower() in name.lower():
            return brand_mapping[brand]
    parts = name.split(" ")
    return " ".join(parts[:2]) if len(parts) >= 2 else parts[0]


def clean_list_string(val):
    """Parse string representation of list from INCI dataset."""
    if pd.isna(val):
        return ""
    try:
        items = ast.literal_eval(val)
        return ", ".join([i.strip() for i in items if i.strip()])
    except:
        return str(val).strip()


def load_and_clean_data():
    """Load all datasets, clean, merge, and return unified dataframes."""

    # Load raw datasets
    skincare_df = pd.read_csv(f"{DATA_DIR}/skincare_products_clean.csv")
    cosmetics_df = pd.read_csv(f"{DATA_DIR}/cosmetics.csv")
    inci_df = pd.read_csv(f"{DATA_DIR}/ingredientsList.csv")

    # Clean skincare_products_clean
    skincare_df["ingredients_clean"] = skincare_df["clean_ingreds"].apply(parse_ingredients_list)
    skincare_df_clean = skincare_df[["product_name", "product_type", "ingredients_clean", "price"]].copy()
    skincare_df_clean.columns = ["name", "product_type", "ingredients", "price"]
    skincare_df_clean["brand"] = "Unknown"
    skincare_df_clean["source"] = "lookfantastic"
    skincare_df_clean["brand"] = skincare_df_clean["name"].apply(lambda x: x.split(" ")[0] if pd.notna(x) else "Unknown")
    for col in ["combination", "dry", "normal", "oily", "sensitive"]:
        skincare_df_clean[col] = None

    # Clean cosmetics dataset
    cosmetics_df["ingredients_clean"] = cosmetics_df["Ingredients"].apply(
        lambda x: ", ".join([i.strip().lower() for i in str(x).split(",")]) if pd.notna(x) else ""
    )
    cosmetics_df_clean = cosmetics_df[["Name", "Label", "ingredients_clean", "Price", "Brand",
                                        "Combination", "Dry", "Normal", "Oily", "Sensitive"]].copy()
    cosmetics_df_clean.columns = ["name", "product_type", "ingredients", "price", "brand",
                                   "combination", "dry", "normal", "oily", "sensitive"]
    cosmetics_df_clean["source"] = "sephora"

    # Merge
    products_df = pd.concat([skincare_df_clean, cosmetics_df_clean], ignore_index=True)

    # Fix brand extraction for lookfantastic products
    mask = products_df["source"] == "lookfantastic"
    products_df.loc[mask, "brand"] = products_df.loc[mask, "name"].apply(extract_brand)

    # Standardize product type names
    product_type_mapping = {
        "Moisturiser": "Moisturizer",
        "Eye Care": "Eye cream",
    }
    exclude_types = ["Body Wash", "Bath Salts", "Bath Oil"]
    products_df = products_df[~products_df["product_type"].isin(exclude_types)].reset_index(drop=True)
    products_df["product_type"] = products_df["product_type"].replace(product_type_mapping)
    products_df = products_df[products_df["ingredients"].str.len() > 0].reset_index(drop=True)

    # Clean INCI
    inci_df["good_for"] = inci_df["who_is_it_good_for"].apply(clean_list_string)
    inci_df["avoid_for"] = inci_df["who_should_avoid"].apply(clean_list_string)
    inci_df["full_description"] = inci_df.apply(lambda row:
        f"Ingredient: {row['name']}. "
        f"Description: {row['short_description'] if pd.notna(row['short_description']) else 'N/A'}. "
        f"Function: {row['what_does_it_do'] if pd.notna(row['what_does_it_do']) else 'N/A'}. "
        f"Good for: {row['good_for']}. "
        f"Avoid if: {row['avoid_for']}.",
        axis=1
    )

    print(f"Loaded {len(products_df)} products and {len(inci_df)} ingredients.")
    return products_df, inci_df


def build_vector_database(products_df, inci_df):
    """Create ChromaDB collections for products and ingredients."""

    chroma_client = chromadb.Client()

    # Product collection
    try:
        chroma_client.delete_collection("skincare_products")
    except:
        pass

    product_collection = chroma_client.create_collection(
        name="skincare_products",
        metadata={"hnsw:space": "cosine"}
    )

    product_documents = []
    product_metadata = []
    product_ids = []

    for idx, row in products_df.iterrows():
        skin_types = []
        for st in ["combination", "dry", "normal", "oily", "sensitive"]:
            if row.get(st) == 1:
                skin_types.append(st)
        skin_type_str = ", ".join(skin_types) if skin_types else "not specified"

        doc = (
            f"Product: {row['name']}. "
            f"Brand: {row['brand']}. "
            f"Type: {row['product_type']}. "
            f"Price: {row['price']}. "
            f"Suitable for skin types: {skin_type_str}. "
            f"Ingredients: {row['ingredients']}."
        )
        product_documents.append(doc)
        product_metadata.append({
            "name": str(row["name"]),
            "brand": str(row["brand"]),
            "product_type": str(row["product_type"]),
            "price": str(row["price"]),
            "source": str(row["source"]),
            "skin_types": skin_type_str
        })
        product_ids.append(f"product_{idx}")

    batch_size = 500
    for i in range(0, len(product_documents), batch_size):
        end = min(i + batch_size, len(product_documents))
        product_collection.add(
            documents=product_documents[i:end],
            metadatas=product_metadata[i:end],
            ids=product_ids[i:end]
        )

    # Ingredient collection
    try:
        chroma_client.delete_collection("skincare_ingredients")
    except:
        pass

    ingredient_collection = chroma_client.create_collection(
        name="skincare_ingredients",
        metadata={"hnsw:space": "cosine"}
    )

    ingredient_ids = []
    idx = 0
    while idx < len(inci_df):
        ingredient_ids.append("ingredient_" + str(idx))
        idx = idx + 1

    ingredient_metadata = []
    for idx, row in inci_df.iterrows():
        ingredient_metadata.append({
            "name": str(row["name"]) if pd.notna(row["name"]) else "",
            "good_for": str(row["good_for"]),
            "avoid_for": str(row["avoid_for"])
        })

    ingredient_collection.add(
        documents=inci_df["full_description"].tolist(),
        metadatas=ingredient_metadata,
        ids=ingredient_ids
    )

    print(f"Indexed {product_collection.count()} products and {ingredient_collection.count()} ingredients.")
    return product_collection, ingredient_collection

Overwriting /content/drive/MyDrive/skincare_project/data_loader.py


## agents/skin_profile.py
Agent 1: Takes raw user text and extracts a structured skin profile using Gemini.

In [58]:
%%writefile /content/drive/MyDrive/skincare_project/agents/skin_profile.py
import json
import re


def skin_profile_agent(client, user_input):
    """Parse raw user text into a structured skin profile using Gemini.

    Args:
        client: Google GenAI client instance.
        user_input: Raw natural language string from the user.

    Returns:
        dict with keys: skin_type, concerns, allergies, age,
        current_products, goals, routine_request.
    """
    prompt = f"""You are a skincare intake specialist. Analyze the following user input and extract
a structured skin profile. Return ONLY valid JSON with no markdown formatting, no backticks, no explanation.

User input: "{user_input}"

Return this exact JSON structure (use null for any field not mentioned):
{{
    "skin_type": "oily/dry/combination/sensitive/normal or null",
    "concerns": ["list of skin concerns mentioned, e.g., acne, dark circles, redness, wrinkles"],
    "allergies": ["list of any allergies or ingredients to avoid"],
    "age": null,
    "current_products": ["list of any products the user currently uses"],
    "goals": ["what the user wants to achieve, e.g., reduce acne, brighten skin"],
    "routine_request": "full routine / specific product / product suggestion"
}}"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    try:
        profile = json.loads(response.text)
    except json.JSONDecodeError:
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            profile = json.loads(json_match.group())
        else:
            profile = {"error": "Could not parse profile", "raw": response.text}

    return profile

Overwriting /content/drive/MyDrive/skincare_project/agents/skin_profile.py


## agents/retrieval.py
Agent 2: Searches the ChromaDB product collection using natural language queries.

In [59]:
%%writefile /content/drive/MyDrive/skincare_project/agents/retrieval.py
import json
import re

def search_products(product_collection, query, n_results=5, product_type=None):
    """Search the product vector database using a natural language query.

    Args:
        product_collection: ChromaDB collection of skincare products.
        query: Natural language search string.
        n_results: Number of products to return.
        product_type: Optional filter (e.g., 'Cleanser', 'Moisturizer').

    Returns:
        ChromaDB query results dict with ids, documents, metadatas, distances.
    """
    filter = {"product_type": product_type} if product_type else None

    results = product_collection.query(
        query_texts=[query],
        n_results=n_results,
        where=filter
    )
    return results

Overwriting /content/drive/MyDrive/skincare_project/agents/retrieval.py


## agents/conflict_checker.py
Agent 3: Checks for ingredient conflicts and beneficial pairs across retrieved products.

In [60]:
%%writefile /content/drive/MyDrive/skincare_project/agents/conflict_checker.py
import json
import re

# Ingredient interaction rules grounded in the Skincare Knowledge Graph dataset
# Rules are split into two categories:
#   - conflict_rules_raw: ingredient combinations to AVOID (adverse effects)
#   - beneficial_pairs: ingredient combinations that WORK WELL together

conflict_rules_raw = [
    {"ingredient_a": "retinol", "ingredient_b": "vitamin c", "reason": "Cancel out effects."},
    {"ingredient_a": "retinol", "ingredient_b": "aha", "reason": "Cancel out effects and cause irritation."},
    {"ingredient_a": "retinol", "ingredient_b": "bha", "reason": "May cause breakouts, dry skin, and irritation."},
    {"ingredient_a": "retinol", "ingredient_b": "citric acid", "reason": "Excessive dryness, redness, sensitivity, or a rash."},
    {"ingredient_a": "retinol", "ingredient_b": "benzoyl peroxide", "reason": "Too harsh for skin and cancel out effects."},
    {"ingredient_a": "aha", "ingredient_b": "niacinamide", "reason": "Can cause redness."},
    {"ingredient_a": "bha", "ingredient_b": "niacinamide", "reason": "Can cause redness."},
    {"ingredient_a": "aha", "ingredient_b": "vitamin c", "reason": "Can cause irritation."},
    {"ingredient_a": "bha", "ingredient_b": "vitamin c", "reason": "Can cause irritation."},
    {"ingredient_a": "salicylic acid", "ingredient_b": "benzoyl peroxide", "reason": "Can cause skin irritation."},
]

beneficial_pairs = [
    {"ingredient_a": "hyaluronic acid", "ingredient_b": "polyglutamic acid", "benefit": "Better hydration."},
    {"ingredient_a": "retinol", "ingredient_b": "niacinamide", "benefit": "Improves skin blemishes, diminishes ageing, and evens out skin tone."},
    {"ingredient_a": "retinol", "ingredient_b": "peptides", "benefit": "Produces collagen and hyaluronic acid, reduces inflammation, and increases cell turnover."},
    {"ingredient_a": "vitamin c", "ingredient_b": "vitamin e", "benefit": "Can help prevent photodamage."},
    {"ingredient_a": "vitamin c", "ingredient_b": "ferulic acid", "benefit": "Ferulic acid stabilizes vitamin C and fends off free radicals."},
]

# Build bidirectional conflict lookup so checking either ingredient finds the rule
conflict_lookup = {}
for rule in conflict_rules_raw:
    a, b = rule["ingredient_a"], rule["ingredient_b"]
    conflict_lookup.setdefault(a, []).append({"conflicts_with": b, "reason": rule["reason"]})
    conflict_lookup.setdefault(b, []).append({"conflicts_with": a, "reason": rule["reason"]})

# Build bidirectional beneficial lookup similarly
beneficial_lookup = {}
for pair in beneficial_pairs:
    a, b = pair["ingredient_a"], pair["ingredient_b"]
    beneficial_lookup.setdefault(a, []).append({"pairs_with": b, "benefit": pair["benefit"]})
    beneficial_lookup.setdefault(b, []).append({"pairs_with": a, "benefit": pair["benefit"]})


def rag_conflict_check(client, ingredient_collection, ingredients_list):
    uncovered = [i for i in ingredients_list if i not in conflict_lookup]
    if not uncovered:
        return []

    # Query ChromaDB per ingredient for better retrieval
    all_docs = []
    for ing in uncovered[:15]:
        res = ingredient_collection.query(query_texts=[ing], n_results=3)
        if res["documents"][0]:
            all_docs.extend(res["documents"][0])

    context = "\n\n".join(all_docs)
    if not context.strip():
        return []

    prompt = f"""You are a cosmetic chemist. Based on the INCI ingredient information below,
identify any combinations from the routine that should NOT be used together.

FULL ROUTINE INGREDIENTS: {ingredients_list}
INGREDIENTS TO FOCUS ON (not covered by hardcoded rules): {uncovered[:15]}

INCI DESCRIPTIONS:
{context}

Return ONLY a valid JSON list. Each conflict must have:
- ingredient_a
- ingredient_b
- reason
- severity: one of "mild", "moderate", "severe"

If no conflicts, return []. No markdown, no explanation, just the JSON list."""

    try:
        response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
        text = re.sub(r'```json|```', '', response.text.strip()).strip()
        rag_conflicts = json.loads(text)

        # Deduplicate (a,b) and (b,a) pairs
        seen = set()
        deduped = []
        for c in rag_conflicts:
            key = frozenset([c["ingredient_a"], c["ingredient_b"]])
            if key not in seen:
                seen.add(key)
                c["source"] = "rag"
                deduped.append(c)

        return deduped

    except json.JSONDecodeError as e:
        print(f"[conflict_checker] Failed to parse Gemini response: {e}")
        return []
    except Exception as e:
        print(f"[conflict_checker] RAG check failed: {e}")
        return []

Overwriting /content/drive/MyDrive/skincare_project/agents/conflict_checker.py


## agents/routine_builder.py
Agent 4: Assembles retrieved products into a structured skincare routine.

In [61]:
%%writefile /content/drive/MyDrive/skincare_project/agents/routine_builder.py
import json
import re
def routine_builder_agent(client, profile, retrieved_products, conflict_report, routine_pref=None):
    """Assemble retrieved products into a structured skincare routine using Gemini.

    Args:
        client: Google GenAI client instance.
        profile: dict from skin_profile_agent (skin_type, concerns, allergies, etc.).
        retrieved_products: list of dicts with keys 'name', 'brand', 'product_type', 'ingredients', 'price'.
        conflict_report: dict with keys 'conflicts' (list) and 'allergy_flags' (list) from conflict checker.

    Returns:
        dict with keys: morning_routine, evening_routine, notes, warnings.
    """

    # Format product list for the prompt
    product_list_str = ""
    for i, p in enumerate(retrieved_products, 1):
        product_list_str += f"{i}. {p['name']} (Brand: {p['brand']}, Type: {p['product_type']}, Price: {p['price']})\n"
        product_list_str += f"   Ingredients: {p['ingredients'][:200]}...\n"

    # Format conflict info
    conflict_str = "None detected." if not conflict_report["conflicts"] else ""
    for c in conflict_report["conflicts"]:
        conflict_str += f"- AVOID: {c['ingredient_a']} + {c['ingredient_b']} — {c['reason']}\n"

    allergy_str = "None detected." if not conflict_report["allergy_flags"] else ""
    for a in conflict_report["allergy_flags"]:
        allergy_str += f"- Product '{a['product']}' contains allergen: {a['allergen']}\n"

    beneficial_str = ""
    if conflict_report.get("beneficial"):
        for b in conflict_report["beneficial"]:
            beneficial_str += f"- {b['ingredient_a']} + {b['ingredient_b']}: {b['benefit']}\n"
    if not beneficial_str:
        beneficial_str = "None identified."

    prompt = f"""You are a skincare routine specialist. Based on the user's skin profile,
retrieved product candidates, and ingredient safety analysis, build a personalized
morning and evening skincare routine.

USER PROFILE:
- Skin type: {profile.get('skin_type') or 'unknown'}
- Concerns: {', '.join(profile.get('concerns') or []) or 'none specified'}
- Allergies: {', '.join(profile.get('allergies') or []) or 'none'}
- Age: {profile.get('age') or 'not specified'}
- Goals: {', '.join(profile.get('goals') or []) or 'general skincare'}
- Routine request: {profile.get('routine_request') or 'full routine'}

AVAILABLE PRODUCTS:
{product_list_str}

INGREDIENT CONFLICTS DETECTED:
{conflict_str}

ALLERGY FLAGS:
{allergy_str}

BENEFICIAL COMBINATIONS:
{beneficial_str}

INSTRUCTIONS:
1. Select appropriate products from the list above for a morning and evening routine.
2. DO NOT include any products flagged for allergy conflicts.
3. If ingredient conflicts were detected, separate conflicting products into different routines
   (e.g., one in morning, one in evening) or exclude one.
4. Order products in standard skincare order, for example: cleanser → toner → serum/treatment → moisturizer → sunscreen (AM only).
5. Explain WHY each product was chosen based on the user's concerns and goals.

Return ONLY valid JSON with no markdown formatting, no backticks, no explanation outside the JSON.
Use this exact structure:
{{
    "morning_routine": [
        {{
            "step": 1,
            "product_type": "Cleanser",
            "product_name": "...",
            "brand": "...",
            "why": "brief explanation of why this product suits the user"
        }}
    ],
    "evening_routine": [
        {{
            "step": 1,
            "product_type": "Cleanser",
            "product_name": "...",
            "brand": "...",
            "why": "brief explanation"
        }}
    ],
    "notes": ["any general skincare tips relevant to this user's profile"],
    "warnings": ["any flagged conflicts or allergies the user should be aware of"]
}}"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    try:
        routine = json.loads(response.text)
    except json.JSONDecodeError:
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            try:
                routine = json.loads(json_match.group())
            except json.JSONDecodeError:
                routine = {"error": "Could not parse routine", "raw": response.text}
        else:
            routine = {"error": "Could not parse routine", "raw": response.text}

    return routine

Overwriting /content/drive/MyDrive/skincare_project/agents/routine_builder.py


## pipeline.py
Chains all four agents together into a single `full_pipeline` function.


In [62]:
%%writefile /content/drive/MyDrive/skincare_project/pipeline.py
import json
import re
from agents.skin_profile import skin_profile_agent
from agents.retrieval import search_products
from agents.conflict_checker import conflict_lookup, beneficial_lookup, rag_conflict_check
from agents.routine_builder import routine_builder_agent


def detect_routine_preference(client, user_input):
    """Use Gemini to detect if user wants AM, PM, or both routines."""

    prompt = f"""A user is asking for skincare advice. Determine if they want a morning routine, evening routine, or both.

USER INPUT: "{user_input}"

Rules:
- If they explicitly mention morning, daytime, AM, or waking up → return AM
- If they explicitly mention night, evening, PM, bedtime, or before bed → return PM
- If they mention both, neither, or just ask for a general/full routine → return both

Return ONLY one word: AM, PM, or both. Nothing else."""

    try:
        response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
        result = response.text.strip()
        if result in ["AM", "PM", "both"]:
            return result
        return "both"
    except Exception as e:
        print(f"[detect_routine_preference] Gemini call failed: {e}")
        return "both"


def run_conflict_check(client, ingredient_collection, products, profile):
    conflicts_found = []
    beneficial_found = []
    allergy_flags = []

    all_ingredients = set()
    for p in products:
        for ing in p["ingredients"].lower().split(", "):
            all_ingredients.add(ing.strip())
    all_ingredients_list = list(all_ingredients)

    # Step 1: hardcoded rules (fast)
    for ing in all_ingredients_list:
        if ing in conflict_lookup:
            for conflict in conflict_lookup[ing]:
                partner = conflict["conflicts_with"]
                if partner in all_ingredients:
                    pair = tuple(sorted([ing, partner]))
                    entry = {"ingredient_a": pair[0], "ingredient_b": pair[1],
                             "reason": conflict["reason"], "source": "rules"}
                    if entry not in conflicts_found:
                        conflicts_found.append(entry)

    # Step 2: RAG fallback for ingredients not in hardcoded rules
    rag_conflicts = rag_conflict_check(client, ingredient_collection, all_ingredients_list)
    for c in rag_conflicts:
        pair = tuple(sorted([c["ingredient_a"], c["ingredient_b"]]))
        entry = {"ingredient_a": pair[0], "ingredient_b": pair[1],
                 "reason": c["reason"], "source": "rag"}
        if entry not in conflicts_found:
            conflicts_found.append(entry)

    # Beneficial pairs
    for ing in all_ingredients_list:
        if ing in beneficial_lookup:
            for pair in beneficial_lookup[ing]:
                partner = pair["pairs_with"]
                if partner in all_ingredients:
                    sorted_pair = tuple(sorted([ing, partner]))
                    entry = {"ingredient_a": sorted_pair[0], "ingredient_b": sorted_pair[1],
                             "benefit": pair["benefit"]}
                    if entry not in beneficial_found:
                        beneficial_found.append(entry)

    # Allergy check
    user_allergies = [a.lower() for a in (profile.get("allergies") or [])]
    for p in products:
        for allergen in user_allergies:
            if allergen in p["ingredients"].lower():
                allergy_flags.append({"product": p["name"], "allergen": allergen})

    return {"conflicts": conflicts_found, "allergy_flags": allergy_flags, "beneficial": beneficial_found}


def full_pipeline(client, product_collection, ingredient_collection, user_input, products_per_type=3):
    """Run the full skincare recommendation pipeline.

    Agent 1 (Skin Profile) -> Agent 2 (Product Retrieval) -> Agent 3 (Conflict Check) -> Agent 4 (Routine Builder)

    Args:
        client: Google GenAI client instance.
        product_collection: ChromaDB collection of skincare products.
        ingredient_collection: ChromaDB collection of INCI ingredients.
        user_input: Raw natural language string from the user.
        products_per_type: Number of products to retrieve per category (default 3).

    Returns:
        dict with keys: profile, retrieved_products, conflict_report, routine, raw_input, routine_preference.
    """

    # Step 0: Detect routine preference before any agent runs
    routine_pref = detect_routine_preference(client, user_input)
    print(f"Routine preference detected: {routine_pref}")

    # Agent 1: Parse user input into structured skin profile
    profile = skin_profile_agent(client, user_input)

    # Agent 2: Retrieve candidate products based on routine preference
    am_types = ["Cleanser", "Serum", "Moisturizer", "Sun protect"]
    pm_types = ["Cleanser", "Serum", "Moisturizer", "Retinol", "Face oil"]

    if routine_pref == "AM":
        product_types = am_types
    elif routine_pref == "PM":
        product_types = pm_types
    else:
        product_types = list(set(am_types + pm_types))  # all unique types for both

    skin_type = profile.get("skin_type", "")
    concerns = " ".join(profile.get("concerns") or [])

    all_products = []
    for ptype in product_types:
        query = f"{skin_type} skin {concerns} {ptype.lower()}"
        results = search_products(product_collection, query, n_results=products_per_type, product_type=ptype)
        for i in range(len(results["ids"][0])):
            meta = results["metadatas"][0][i]
            all_products.append({
                "name": meta["name"],
                "brand": meta["brand"],
                "product_type": meta["product_type"],
                "ingredients": results["documents"][0][i],
                "price": meta["price"],
                # Tag each product with its intended time of use
                "routine_time": "AM" if ptype == "Sun protect" else
                               "PM" if ptype in ["Retinol", "Face oil"] else "both"
            })

    # Agent 3: Check for conflicts and allergies
    conflict_report = run_conflict_check(client, ingredient_collection, all_products, profile)

    # Agent 4: Build the routine (passes routine_pref so Gemini structures AM/PM correctly)
    routine = routine_builder_agent(client, profile, all_products, conflict_report, routine_pref=routine_pref)

    return {
        "raw_input": user_input,
        "profile": profile,
        "retrieved_products": all_products,
        "conflict_report": conflict_report,
        "routine": routine,
        "routine_preference": routine_pref
    }

Overwriting /content/drive/MyDrive/skincare_project/pipeline.py


## Verify Files Created

In [63]:
base = "/content/drive/MyDrive/skincare_project"
expected_files = [
    "data_loader.py",
    "pipeline.py",
    "agents/__init__.py",
    "agents/skin_profile.py",
    "agents/retrieval.py",
    "agents/conflict_checker.py",
    "agents/routine_builder.py",
]

print("File verification:")
for f in expected_files:
    path = os.path.join(base, f)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {status} - {f}")

print("\nAll modules built! Open demo.ipynb to run the pipeline.")

File verification:
  OK - data_loader.py
  OK - pipeline.py
  OK - agents/__init__.py
  OK - agents/skin_profile.py
  OK - agents/retrieval.py
  OK - agents/conflict_checker.py
  OK - agents/routine_builder.py

All modules built! Open demo.ipynb to run the pipeline.
